# Micro-Hay state-information probe 09

This is a diagnostic, not a deployable surrogate. It compares identical standard linear/MLP probes using either `(frozen GRU hidden + current spike input)` or `(previous true 61-state vector + current spike input)`. Teacher state appears only in the oracle diagnostic arm. The result decides whether explicit predicted-state feedback is the next justified inductive bias.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()): raise RuntimeError(f'{ROOT} exists but is not the project')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## Inputs and run
Attach the existing `hay_micro_4c_event_enriched_v2.h5` and original `gru_mse.pt`. No previous gated-TCN checkpoint is needed. Extraction and every probe epoch display progress and ETA.

In [ ]:
def find_kaggle_input(working_path, pattern, label):
    working_path = Path(working_path)
    candidates = [working_path] if working_path.exists() else []
    if Path('/kaggle/input').exists(): candidates += sorted(Path('/kaggle/input').rglob(pattern))
    if not candidates: raise FileNotFoundError(f'{label} non trovato: {pattern}')
    print(label + ':', candidates[0])
    return candidates[0]
DATASET = find_kaggle_input('/kaggle/working/hay_micro_4c_event_enriched_v2.h5', 'hay_micro_4c_event_enriched_v2*.h5', 'Dataset')
BASELINE = find_kaggle_input('/kaggle/working/gru_mse.pt', 'gru_mse.pt', 'GRU-MSE')
os.environ['HAY_PROBE_DATASET'] = str(DATASET)
os.environ['HAY_PROBE_BASELINE'] = str(BASELINE)
os.environ['HAY_PROBE_OUTPUT'] = '/kaggle/working/hay_micro_state_information_probe_09'
os.environ['HAY_PROBE_EPOCHS'] = '15'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_state_information_probe_09.py

## Evidence
For classification, compare test average precision/F1 under the natural class prior. For regression, compare support/core voltage RMSE. A large advantage for `state` justifies predicted-state feedback; similar results would falsify that hypothesis.

In [ ]:
import pandas as pd
from IPython.display import Image, display
RESULTS = Path('/kaggle/working/hay_micro_state_information_probe_09')
classification = pd.read_csv(RESULTS / 'classification_probe.csv')
display(classification.sort_values(['split','average_precision'], ascending=[True,False]))
regression = pd.read_csv(RESULTS / 'voltage_residual_probe.csv')
display(regression.sort_values(['split','region','probe_rmse_mV']))
display(Image(str(RESULTS / 'precision_recall.png')))

## Export
Send the resulting ZIP back for analysis. It contains metrics, plot, probe checkpoints and the experiment contract.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, FileLink, display
zip_path = Path(make_archive('/kaggle/working/hay_micro_state_information_probe_09_complete', 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{encoded}'),x=new Uint8Array(b.length);for(let i=0;i<b.length;i++)x[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([x],{{type:'application/zip'}})),a=document.createElement('a');a.href=u;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),60000);"""))
print('Download avviato:', zip_path.name)